# 🦙 LlamaIndex Master Course - Level 4
## Embeddings, Vector Stores & Indexing

Welcome to Level 4! In this notebook, we'll cover configuring embedding models, using local persistence with StorageContext, managing external vector stores like ChromaDB, and updating documents within an active index.

In [ ]:
# Setup - Run this cell first!
!pip install -q llama-index-core llama-index-embeddings-openai llama-index-embeddings-huggingface llama-index-vector-stores-chroma chromadb

import os
from dotenv import load_dotenv

load_dotenv() # Load your OpenAI API key if you have one

print("Libraries loaded successfully!")

### 1. Generating Embeddings

Embeddings are vectors that represent semantic meaning. Let's inspect an embedding generated directly from a model.

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding

# Note: This requires the OPENAI_API_KEY environment variable to be set.
embed_model = OpenAIEmbedding(model="text-embedding-3-small")

vector = embed_model.get_text_embedding("TechNova Q4 Revenue was $7M.")

print(f"Vector Dimensions: {len(vector)}")
print(f"First 5 values: {vector[:5]}")

### 2. Local Index Persistence

By default, `VectorStoreIndex` stores data in memory. Here is how you persist it to disk and reload it to avoid re-embedding costs.

In [ ]:
from llama_index.core import VectorStoreIndex, Document, StorageContext, load_index_from_storage

PERSIST_DIR = "./storage_level4"

if not os.path.exists(PERSIST_DIR):
    print("Building index from scratch...")
    docs = [Document(text="TechNova makes AI software.")]
    index = VectorStoreIndex.from_documents(docs)
    
    # Save to disk
    index.storage_context.persist(persist_dir=PERSIST_DIR)
    print(f"Saved to {PERSIST_DIR}.")
    
else:
    print("Loading index from disk...")
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)
    print("Loaded successfully!")

### 3. ChromaDB & Document Updates

Let's wire up a local Chroma database and perform an insert, update, and query.

In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import Settings

Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# 1. Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path="./chroma_test_db")
collection = chroma_client.get_or_create_collection("level4_demo")

# 2. Wire up to LlamaIndex
vector_store = ChromaVectorStore(chroma_collection=collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 3. Initial Ingestion
docs = [
    Document(
        text="Employee handbook: Unlimited PTO requires manager approval.",
        doc_id="hr_handbook"
    )
]

print("Ingesting initial document...")
index_chroma = VectorStoreIndex.from_documents(docs, storage_context=storage_context)

# 4. Query Initial Data
print("\n--- Query Before Update ---")
response = index_chroma.as_query_engine().query("What is the PTO policy?")
print(response)

# 5. Update the Document
print("\nUpdating document...")
new_hr_doc = Document(
    text="Employee handbook UPDATE 2024: Unlimited PTO is replaced with 30 days fixed PTO.",
    doc_id="hr_handbook" # Must match exactly to update!
)
index_chroma.update_ref_doc(new_hr_doc)

# 6. Query Updated Data
print("\n--- Query After Update ---")
response_after = index_chroma.as_query_engine().query("What is the PTO policy?")
print(response_after)

### 🎉 Level 4 Complete!
You have successfully mastered embeddings, storage contexts, and dynamic index updates! We are now ready to tackle Retrievers & Query Engines in Level 5.